In [1]:
import argparse
import functools
import gc
import itertools
import logging
import math
import os
from distutils.util import strtobool
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import (
    DistributedDataParallelKwargs,
    ProjectConfiguration,
    set_seed,
)
from huggingface_hub import create_repo, upload_folder
from huggingface_hub.utils import insecure_hashlib
from packaging import version
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from transformers import AutoTokenizer, PretrainedConfig

import diffusers
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionXLPipeline,
)

from diffusers.loaders import LoraLoaderMixin
from diffusers.optimization import get_scheduler
from diffusers.utils import check_min_version, is_wandb_available
from diffusers.utils.import_utils import is_xformers_available

from unziplora_unet.unziplora_linear_layer import UnZipLoRALinearLayer
from unziplora_unet.pipeline_stable_diffusion_xl import StableDiffusionXLUnZipLoRAPipeline
from unziplora_unet.unet_2d_condition import UNet2DConditionModel
from unziplora_unet.utils import *

/home/xzh/miniconda3/envs/unziplora/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/home/xzh/miniconda3/envs/unziplora/lib/python3.11/site-packages/diffusers/utils/outputs.py:63: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [2]:
unet = UNet2DConditionModel.from_pretrained(
    '/home/xzh/xzh/pretrained/sd_xl_base_1.0', subfolder='unet'
)
unet

UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): LoRACompatibleLinear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): LoRACompatibleLinear(in_features=1280, out_features=1280, bias=True)
  )
  (add_time_proj): Timesteps()
  (add_embedding): TimestepEmbedding(
    (linear_1): LoRACompatibleLinear(in_features=2816, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): LoRACompatibleLinear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): DownBlock2D(
      (resnets): ModuleList(
        (0-1): 2 x ResnetBlock2D(
          (norm1): GroupNorm(32, 320, eps=1e-05, affine=True)
          (conv1): LoRACompatibleConv(320, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (time_emb_proj): LoRACompatibleLinear(in_features=1280, out_features=320, bias=True)
          (

In [3]:
for attn_processor_name, attn_processor in unet.attn_processors.items():
    attn_module = unet
    print(attn_processor_name , attn_processor)

down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2aec8350>
down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2af306d0>
down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2ae9b2d0>
down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2aefa990>
down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2ae99650>
down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2aec8e50>
down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2b05f910>

In [4]:
unet.attn_processors

{'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8350>,
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2af306d0>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae9b2d0>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aefa990>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae99650>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8e50>,
 'down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2b05f910>,
 'down_blocks

In [5]:
from unziplora_unet.utils import attach_recorders_to_unet
original_procs = unet.attn_processors
original_procs

{'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8350>,
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2af306d0>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae9b2d0>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aefa990>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae99650>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8e50>,
 'down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2b05f910>,
 'down_blocks

In [6]:
attach_recorders_to_unet(unet, [1,2], 64)

([<unziplora_unet.utils.RecordingCrossAttnProcessor at 0x7f6f2abbe490>,
 {'down_blocks.1.attentions.0.transformer_blocks.0.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2af306d0>,
  'down_blocks.1.attentions.0.transformer_blocks.1.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aefa990>,
  'down_blocks.1.attentions.1.transformer_blocks.0.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8e50>,
  'down_blocks.1.attentions.1.transformer_blocks.1.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae9a310>,
  'down_blocks.2.attentions.0.transformer_blocks.0.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2af30a10>,
  'down_blocks.2.attentions.0.transformer_blocks.1.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f4022d310>,
  'down_blocks.2.attentions.0.transformer_blocks.2.attn2': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f4022c190>,
  'd

In [8]:
unet.set_attn_processor(original_procs)

In [7]:
unet.attn_processors.items()

dict_items([('down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2aec8350>), ('down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor', <unziplora_unet.utils.RecordingCrossAttnProcessor object at 0x7f6f2abbe490>), ('down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2ae9b2d0>), ('down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor', <unziplora_unet.utils.RecordingCrossAttnProcessor object at 0x7f6f2abbeed0>), ('down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor', <unziplora_unet.attention_processor.AttnProcessor2_0 object at 0x7f6f2ae99650>), ('down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor', <unziplora_unet.utils.RecordingCrossAttnProcessor object at 0x7f6f2acca890>), ('down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor', <unziplora_unet.attention_processor

In [9]:
unet.attn_processors

{'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8350>,
 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2af306d0>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae9b2d0>,
 'down_blocks.1.attentions.0.transformer_blocks.1.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aefa990>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2ae99650>,
 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2aec8e50>,
 'down_blocks.1.attentions.1.transformer_blocks.1.attn1.processor': <unziplora_unet.attention_processor.AttnProcessor2_0 at 0x7f6f2b05f910>,
 'down_blocks

In [38]:
for name, proc in unet.attn_processors.items():
    
    attn_module = unet
    print(name)

ValueError: Make sure that either all layers or no layers have LoRA activated, but have {'to_q': True, 'to_k': False, 'to_v': False, 'to_out.0': False}

In [10]:
for name, proc in unet.attn_processors.items():
    
    attn_module = unet
    for n in name.split('.')[:-1]:
        attn_module = getattr(attn_module, n)
    attn_module.to_q.set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
        )
    )
    attn_module.to_k.set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
        )
    )
    attn_module.to_v.set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
        )
    )
    attn_module.to_out[0].set_lora_layer(
        UnZipLoRALinearLayer(
            in_features=attn_module.to_q.in_features,
            out_features=attn_module.to_q.out_features,
            rank=64,
            lora_matrix_num = 2,
            device='cuda',
            # dtype=weight_dtype,
            lora_matrix_key = ["content", "style"],
        )
    )


In [42]:
for _name, _mod in unet.named_modules():
    if not _name.endswith(".attn2"):
        continue
    print(_name)
for _name, _mod in unet.named_modules():
    if ".attn2" not in _name:
        continue
    print(_name)

down_blocks.1.attentions.0.transformer_blocks.0.attn2
down_blocks.1.attentions.0.transformer_blocks.1.attn2
down_blocks.1.attentions.1.transformer_blocks.0.attn2
down_blocks.1.attentions.1.transformer_blocks.1.attn2
down_blocks.2.attentions.0.transformer_blocks.0.attn2
down_blocks.2.attentions.0.transformer_blocks.1.attn2
down_blocks.2.attentions.0.transformer_blocks.2.attn2
down_blocks.2.attentions.0.transformer_blocks.3.attn2
down_blocks.2.attentions.0.transformer_blocks.4.attn2
down_blocks.2.attentions.0.transformer_blocks.5.attn2
down_blocks.2.attentions.0.transformer_blocks.6.attn2
down_blocks.2.attentions.0.transformer_blocks.7.attn2
down_blocks.2.attentions.0.transformer_blocks.8.attn2
down_blocks.2.attentions.0.transformer_blocks.9.attn2
down_blocks.2.attentions.1.transformer_blocks.0.attn2
down_blocks.2.attentions.1.transformer_blocks.1.attn2
down_blocks.2.attentions.1.transformer_blocks.2.attn2
down_blocks.2.attentions.1.transformer_blocks.3.attn2
down_blocks.2.attentions.1.t

#### 直接用 Attention Block 来结尾的 Block 进行设置来捕捉 content capture

In [ ]:
for _name, _mod in unet.named_modules():
    if not _name.endswith(".attn2"):
        continue
    print(_name)
    for _layer in [_mod.to_k, _mod.to_v]:
        if getattr(_layer, "lora_layer", None) is None:
            continue
        kind = "k" if _layer is _mod.to_k else "v"
        _layer.lora_layer.register_forward_hook(_make_content_lora_hook(kind))
    _mod.to_k.register_forward_hook(_mask_base_hook("k"))


down_blocks.1.attentions.0.transformer_blocks.0.attn2


NameError: name '_make_content_lora_hook' is not defined